# Per-Layer Dead Residual Alpha Probe (滚动训练 + OOS)

全量数据 2020-2026, 2024-06 起为 OOS, 之前滚动半年 fold.


In [ ]:
# 1. 加载 + 投影 + 收益 + 滚动切分 + h_cache
import os, sys, time, glob, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

ROOT = "/home/intern_fjq_2026/Projects/chinese-wwm-roberta"
os.chdir(ROOT); sys.path.insert(0, ROOT)

# --- 加载 per_file (全量, 不按年份过滤) ---
per_dir = os.path.join(ROOT, "artifacts", "gubapost_cls", "per_file")
pf_files = sorted(glob.glob(os.path.join(per_dir, "*.parquet")))
assert pf_files, "per_file/ 下没有文件"
print(f"加载 {len(pf_files)} 个 per-file parquet...")
NEED_COLS = ["available_date", "symbol", "n_posts", "sum_cls"] + [f"sum_attn_{L:02d}" for L in range(1, 13)]
dfs = [pd.read_parquet(f, columns=NEED_COLS) for f in pf_files]
pf = pd.concat(dfs, ignore_index=True); del dfs
nposts = pf["n_posts"].values.astype(np.float64)
nposts_safe = np.where(nposts > 0, nposts, 1.0)
cls = np.stack(pf["sum_cls"].values).astype(np.float32)
cls_df = pf[["available_date", "symbol", "n_posts"]].copy()
N_LAYERS = 12; DEAD_DIM = 128
dead = {}
for L in range(1, N_LAYERS+1):
    dead[L] = np.stack(pf[f"sum_attn_{L:02d}"].values).astype(np.float32)
print(f"dead-128: {N_LAYERS} layers, shape={dead[1].shape}")
del pf
print(f"CLS: {cls.shape} | dates: {cls_df.available_date.min()}~{cls_df.available_date.max()}")

# --- 收益 (前移1天, winsorize) ---
rtn = pd.read_parquet("/home/intern_fjq_2026/data/RTN_daily/rtn_1d.parquet")
rtn_long = rtn.melt(id_vars="date", var_name="sym", value_name="r")
rtn_long["symbol"] = rtn_long["sym"].str.split(".").str[0]
rtn_long["date"] = pd.to_datetime(rtn_long["date"]).dt.strftime("%Y-%m-%d")
rtn_long = rtn_long.sort_values(["symbol", "date"])
rtn_long["y"] = rtn_long.groupby("symbol")["r"].shift(-1)
rtn_long = rtn_long[["date", "symbol", "y"]].dropna(subset=["y"])
rtn_long["y"] = rtn_long["y"].clip(-0.2, 0.2)

merged = cls_df[["available_date", "symbol"]].merge(
    rtn_long, left_on=["available_date", "symbol"], right_on=["date", "symbol"], how="inner")
merged = merged[["available_date", "symbol", "y"]].reset_index(drop=True)
merged["ym"] = merged["available_date"].str[:7]

# 对齐 features
meta_key = cls_df["available_date"] + "_" + cls_df["symbol"]
merged_key = merged["available_date"] + "_" + merged["symbol"]
mask = meta_key.isin(set(merged_key)).values
cls = cls[mask]
nposts_aligned = nposts_safe[mask]
for L in range(1, N_LAYERS+1):
    dead[L] = dead[L][mask]
assert len(cls) == len(merged), f"{len(cls)} != {len(merged)}"

y_all = merged["y"].values.astype(np.float64)
dates_all = merged["available_date"].values
idx_all = np.arange(len(merged))

# --- 滚动切分: in-sample (< 2024-06) + OOS (>= 2024-06) ---
OOS_START = "2024-06"
is_oos = merged["ym"] >= OOS_START
is_insample = ~is_oos
insample_idx = idx_all[is_insample.values]
oos_idx = idx_all[is_oos.values]

def ym_to_half(ym):
    y, m = ym.split("-")
    return y + ("H1" if int(m) <= 6 else "H2")

insample_ym = merged.loc[is_insample, "ym"].values
half_periods = sorted(set(ym_to_half(ym) for ym in insample_ym))
folds = []
for i in range(1, len(half_periods)):
    train_periods = set(half_periods[:i])
    test_period = half_periods[i]
    train_mask = np.array([ym_to_half(ym) in train_periods for ym in insample_ym])
    test_mask = np.array([ym_to_half(ym) == test_period for ym in insample_ym])
    ti = insample_idx[train_mask]
    ei = insample_idx[test_mask]
    if len(ei) >= 20:
        folds.append((ti, ei, test_period))
        print(f"  fold {test_period}: train={len(ti):,} test={len(ei):,}")
print(f"\nIn-sample folds: {len(folds)} | OOS: {len(oos_idx):,} samples (>= {OOS_START})")

# --- helpers ---
ALPHA = 100
def ric(yp, yt, d):
    ics = []
    for dt in np.unique(d):
        m = d == dt
        if m.sum() >= 10:
            ics.append(spearmanr(yp[m], yt[m])[0])
    return np.array(ics)
def rfp(Xtr, ytr, Xte):
    sc = StandardScaler()
    m = Ridge(alpha=ALPHA)
    m.fit(sc.fit_transform(Xtr), ytr)
    return m.predict(sc.transform(Xte))
def safe_icir(a):
    s = a.std()
    return a.mean() / s if s > 1e-6 else 0

# --- H 方向 + h_cache ---
DIRS_PATH = os.path.join(ROOT, "artifacts", "checkpoint_activation_rank", "runs", "gubapost_v1",
                         "extensions", "coverage_ablation_v1", "direction_sets.npz")
dirs_npz = np.load(DIRS_PATH)
H = cls @ dirs_npz["keep_317_complement_K64"].astype(np.float32)
L_cov = cls @ dirs_npz["keep_451_lowcov_K64"].astype(np.float32)
print(f"H {H.shape} L {L_cov.shape}")

def build_h_cache(train_idx, test_idx):
    """5-fold CV 残差 (train) + Ridge 预测残差 (test)."""
    kf = KFold(5, shuffle=False)
    oos = np.full(len(train_idx), np.nan)
    for a, b in kf.split(H[train_idx]):
        sc = StandardScaler(); m = Ridge(alpha=ALPHA)
        m.fit(sc.fit_transform(H[train_idx][a]), y_all[train_idx][a])
        oos[b] = m.predict(sc.transform(H[train_idx][b]))
    train_res = y_all[train_idx] - oos
    ypH = rfp(H[train_idx], y_all[train_idx], H[test_idx])
    test_res = y_all[test_idx] - ypH
    return train_res, test_res, dates_all[test_idx]

# in-sample folds h_cache
h_cache = {}
for fi, (ti, ei, period) in enumerate(folds):
    h_cache[fi] = (*build_h_cache(ti, ei), ti, ei)
    print(f"  h_cache fold {period} done")
# OOS h_cache (train = 全 in-sample, test = OOS)
h_cache_oos = (*build_h_cache(insample_idx, oos_idx), insample_idx, oos_idx)
print("h_cache (in-sample + OOS) 完成")

In [ ]:
# 2. Per-layer probe: in-sample folds + OOS
layer_results = []
for L in range(1, N_LAYERS+1):
    t0 = time.time()
    r_L = dead[L]

    # In-sample folds IC
    ic_f = []
    for fi, (ti, ei, period) in enumerate(folds):
        _, _, _, _, _ = h_cache[fi]
        yp = rfp(r_L[ti], y_all[ti], r_L[ei])
        ic_f.append(ric(yp, y_all[ei], dates_all[ei]))
    ic_ins = np.concatenate(ic_f).mean()

    # In-sample residual IC
    res_f = []
    for fi, (ti, ei, period) in enumerate(folds):
        train_res, test_res, test_dates, _, _ = h_cache[fi]
        rp = rfp(r_L[ti], train_res, r_L[ei])
        res_f.append(ric(rp, test_res, test_dates))
    res_ic_ins = np.concatenate(res_f).mean()

    # OOS residual IC
    train_res_oos, test_res_oos, oos_dates, _, _ = h_cache_oos
    rp_oos = rfp(r_L[insample_idx], train_res_oos, r_L[oos_idx])
    oos_ics = ric(rp_oos, test_res_oos, oos_dates)
    res_ic_oos = oos_ics.mean()

    # Random baseline (OOS, 20 random 128-dim)
    rng = np.random.default_rng(L)
    rand_means = []
    for s in range(20):
        Qr = np.linalg.qr(rng.standard_normal((768, DEAD_DIM)))[0].astype(np.float32)
        R = cls @ Qr
        rp = rfp(R[insample_idx], train_res_oos, R[oos_idx])
        rand_means.append(ric(rp, test_res_oos, oos_dates).mean())
    rand_m, rand_s = np.mean(rand_means), np.std(rand_means)
    sig = "YES" if res_ic_oos > rand_m + 2*rand_s else ("~" if res_ic_oos > rand_m else "no")

    layer_results.append({
        "layer": f"L{L:02d}", "IC_ins": ic_ins, "ResIC_ins": res_ic_ins,
        "ResIC_oos": res_ic_oos, "Rand_m": rand_m, "Rand_s": rand_s, "sig": sig,
    })
    print(f"L{L:02d}: IC_ins={ic_ins:.4f} ResIC_ins={res_ic_ins:.4f} "
          f"ResIC_oos={res_ic_oos:.4f} Rand={rand_m:.4f}+/-{rand_s:.4f} {sig} ({time.time()-t0:.0f}s)")
    del r_L

In [ ]:
# 3. 汇总 + 保存 + 图
import matplotlib.pyplot as plt
res_df = pd.DataFrame(layer_results)
print("=== Per-Layer Dead Residual Alpha Probe (rolling + OOS) ===")
print(res_df.to_string(index=False))
n_sig = (res_df.sig == "YES").sum()
print(f"\nOOS 显著层数 (ResIC > Rand+2sigma): {n_sig}/{N_LAYERS}")
res_df.to_parquet(os.path.join(ROOT, "artifacts", "gubapost_cls", "dead_residual_alpha_results.parquet"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(N_LAYERS)
colors = ["red" if s=="YES" else ("orange" if s=="~" else "gray") for s in res_df.sig]
axes[0].bar(x, res_df.ResIC_oos, color=colors, label="ResIC OOS")
axes[0].errorbar(x, res_df.Rand_m, yerr=res_df.Rand_s, fmt="none", ecolor="blue", capsize=3, label="Random")
axes[0].set_xticks(x); axes[0].set_xticklabels(res_df.layer, rotation=45)
axes[0].set_ylabel("mean daily Rank IC"); axes[0].set_title("Per-Layer ResIC OOS: Dead 128 vs Random")
axes[0].axhline(0, color="black", lw=0.5); axes[0].legend(fontsize=8)

axes[1].bar(x, res_df.ResIC_ins, color="steelblue", label="ResIC in-sample")
axes[1].bar(x, res_df.ResIC_oos, color="red", alpha=0.5, label="ResIC OOS")
axes[1].set_xticks(x); axes[1].set_xticklabels(res_df.layer, rotation=45)
axes[1].set_title("In-sample vs OOS ResIC"); axes[1].axhline(0, color="black", lw=0.5)
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()